<a href="https://colab.research.google.com/github/tousif47/Geostormcaster/blob/main/01_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Data Preparation: Model A (t+3h storm forecast, hourly data)

Builds training/validation/test windows from `omni_clean.csv`.

- **Sample** = 24h solar wind window (6 physics-informed features) ending
  at hour *t*, labeled with the exact Kp value of the *next* 3-hour block
- **Labels** = raw Kp values (float, 0.0-9.0); experiment notebooks apply
  their own threshold (Kp 5 = NOAA G1)
- **Splits** = time-ordered: train <=2010, val 2011-2018, test 2019-present
  (random splits would leak the solar cycle)
- **Output** = `windows_a.npz` + `windows_a_norm.json` (normalization uses
  train statistics only)

In [6]:
import pandas as pd
import numpy as np
import json
from numpy.lib.stride_tricks import sliding_window_view

CSV     = '/content/drive/MyDrive/temp/omni_clean.csv'
OUT_DIR = '/content/drive/MyDrive/temp'

## Load and clean

In [7]:
df = pd.read_csv(CSV, parse_dates=['timestamp']).set_index('timestamp').sort_index()
df['kp'] = df['kp'].clip(upper=9)
df['log_temp'] = np.log10(df['temp'].replace(0, np.nan))   # 3K..6.6M K -> ~3.5..6.8

# physics-informed coupling features (raw solar wind -> storm-driving quantities)
bt = df['b_magnitude']
clock = np.degrees(np.arctan2(df['by_gsm'], df['bz_gsm'])) % 360   # IMF clock angle, 0-360
south = np.sin(np.radians(clock) / 2) ** (8 / 3)                  # peaks at 180deg = due southward

df['newell'] = df['speed']**(4/3) * np.maximum(bt, 0)**(2/3) * south   # Newell coupling rate
df['rect_v'] = df['speed'] * np.clip(-df['bz_gsm'], 0, None)           # rectified solar wind speed
df['pressure'] = df['density'] * df['speed']**2                        # dynamic pressure proxy

print(len(df), 'hours:', df.index[0], '->', df.index[-1])

561024 hours: 1963-01-01 00:00:00 -> 2026-12-31 23:00:00


## Labels: Kp of the *next* 3-hour block

Kp is a 3-hourly index. Each hour gets the Kp of the next block boundary,
which makes the prediction lead time exact (always the upcoming block).

In [8]:
kp = df['kp'].values
h = df.index.hour.values

next_off = (3 - (h % 3)) % 3
next_off[next_off == 0] = 3                    # boundary hours still predict the NEXT block
label_idx = np.arange(len(df)) + next_off

kp_next = np.full(len(df), np.nan)
ok = label_idx < len(df)
kp_next[ok] = kp[label_idx[ok]]
df['kp_next'] = kp_next

## Build windows: 24h x 6 features

Small gaps (<=3h) are linearly interpolated — solar wind varies slowly at
hourly scale, so this is safe and recovers the sparse pre-1995 coverage.
Windows with larger gaps are still dropped (strict).

In [9]:
FEATS_A = ['bz_gsm', 'speed', 'density', 'newell', 'rect_v', 'pressure']
raw = pd.DataFrame(df[FEATS_A].values.astype(np.float32),
                   columns=FEATS_A, index=df.index)

data = raw.interpolate(limit=3, limit_direction='both').values.astype(np.float32)

W = sliding_window_view(data, (24, len(FEATS_A))).squeeze(1)   # (N-23, 24, 6)
y_a = kp_next[23:]
t_a = df.index[23:]

mask = ~np.isnan(W).any(axis=(1, 2)) & ~np.isnan(y_a)
X_a, y_a, t_a = W[mask], y_a[mask], t_a[mask]
print(f"windows: {len(X_a)}  (dropped {mask.size - mask.sum()} incomplete)")

windows: 375621  (dropped 185380 incomplete)


## Labels and time splits

Labels are the **exact Kp value** of the next 3-hour block (float, 0.0-9.0).
Experiment notebooks apply their own storm threshold from these values
(Kp 5 = NOAA G1) — storing values instead of class ids keeps every
threshold choice available from a single npz.

- **Splits** = time-ordered: train <=2010, val 2011-2018, test 2019-present
  (random splits would leak the solar cycle; test = "the real future")
- **Normalization** = train statistics only, applied to all splits
- **Output** = `windows_a.npz` + `windows_a_norm.json`

In [10]:
tr = t_a <  '2011-01-01'
va = (t_a >= '2011-01-01') & (t_a < '2019-01-01')
te = t_a >= '2019-01-01'

XA = dict(Xtr=X_a[tr], Xva=X_a[va], Xte=X_a[te],
          ytr=y_a[tr], yva=y_a[va], yte=y_a[te],
          ttr=t_a[tr], tva=t_a[va], tte=t_a[te])

for part in ['tr', 'va', 'te']:
    y = XA['y' + part]
    print(f"{part:3s}: n={len(y):6d}   "
          f"G1+ share (Kp>=5): {(y >= 5).mean():.3f}   "
          f"Kp>=6 share: {(y >= 6).mean():.3f}")

tr : n=240312   G1+ share (Kp>=5): 0.043   Kp>=6 share: 0.014
va : n= 69349   G1+ share (Kp>=5): 0.022   Kp>=6 share: 0.005
te : n= 65960   G1+ share (Kp>=5): 0.023   Kp>=6 share: 0.008


## Normalize (train statistics only) and save

Scaling uses train mean/std applied to all splits — computing them on the
full dataset would leak test information. Constants are saved to JSON
because the same numbers must be baked into the ESP32 firmware later.

In [11]:
flat = XA['Xtr'].reshape(-1, len(FEATS_A))          # all train timesteps pooled
mu, sd = flat.mean(axis=0), flat.std(axis=0)
sd = np.where(sd < 1e-6, 1.0, sd)                    # guard against constant columns

def sc(x):
    return (x - mu) / sd

XA = {k: (sc(v) if k.startswith('X') else v) for k, v in XA.items()}

np.savez_compressed(f'{OUT_DIR}/windows_a.npz', **XA)
with open(f'{OUT_DIR}/windows_a_norm.json', 'w') as f:
    json.dump({'mean': mu.tolist(), 'std': sd.tolist(), 'features': FEATS_A}, f, indent=1)

print('saved windows_a.npz + windows_a_norm.json')

saved windows_a.npz + windows_a_norm.json


## Next

- `02_model_a_t3h.ipynb` loads `windows_a.npz` and applies the G1+ threshold
- `03_model_b_t24h.ipynb` builds its own tabular features from
  `omni_clean.csv` (self-contained, does not use this npz)